In [ ]:
# Cell 1: imports and load
import pandas as pd
import matplotlib.pyplot as plt
from job_shop_lib.benchmarking import load_all_benchmark_instances, load_benchmark_instance
from job_shop_lib.dispatching.rules import DispatchingRuleSolver

all_instances = load_all_benchmark_instances()
taillard = {k: v for k, v in all_instances.items() if k.startswith("ta")}
print(len(taillard), "Taillard instances")

In [ ]:
# Cell 2: what is in one instance
inst = load_benchmark_instance("ta01")
print(inst.name, inst.num_jobs, "jobs x", inst.num_machines, "machines,", inst.num_operations, "operations")
print(inst.metadata)
print("first job:", inst.jobs[0][:4])
op = inst.jobs[0][0]
print("op fields:", op.job_id, op.position_in_job, op.machine_id, op.duration)

In [ ]:
# Cell 3: size and bound summary across all 80 instances
rows = [
    {
        "name": k,
        "jobs": v.num_jobs,
        "machines": v.num_machines,
        "optimum": v.metadata.get("optimum"),
        "lower_bound": v.metadata.get("lower_bound"),
        "upper_bound": v.metadata.get("upper_bound"),
    }
    for k, v in taillard.items()
]
summary = pd.DataFrame(rows)
summary["size"] = summary["jobs"].astype(str) + "x" + summary["machines"].astype(str)
summary["proven_optimal"] = summary["optimum"].notna()
print(summary.groupby("size").agg(n=("name", "count"), proven=("proven_optimal", "sum")))

In [ ]:
# Cell 4: trivial lower bounds vs the published optimum (15x15 and 20x15)
def trivial_lower_bound(instance) -> int:
    return max(max(instance.job_durations), max(instance.machine_loads))

subset = summary[summary["size"].isin(["15x15", "20x15"])].copy()
subset["trivial_lb"] = [trivial_lower_bound(taillard[n]) for n in subset["name"]]
subset["ref"] = subset["optimum"].fillna(subset["upper_bound"])
subset["slack_pct"] = 100 * (subset["ref"] - subset["trivial_lb"]) / subset["ref"]
print(subset[["name", "size", "trivial_lb", "ref", "slack_pct"]].head(12))

In [ ]:
# Cell 5: library dispatching baselines (Day 2 gate reference)
RULES = {
    "SPT": "shortest_processing_time",
    "LPT": "largest_processing_time",
    "FIFO": "first_come_first_served",
}

def makespan_of(instance, rule: str, **solver_kwargs) -> int:
    return DispatchingRuleSolver(rule, **solver_kwargs).solve(instance).makespan()

names = [f"ta{i:02d}" for i in range(1, 11)]
rows = []
for n in names:
    inst = taillard[n]
    ref = inst.metadata.get("optimum") or inst.metadata.get("upper_bound")
    row = {"name": n, "ref": ref}
    for label, rule in RULES.items():
        row[label] = makespan_of(inst, rule)
        row[f"{label}_gap%"] = round(100 * (row[label] - ref) / ref, 1)
    rows.append(row)
baselines = pd.DataFrame(rows)
print(baselines)

In [ ]:
# Cell 6: does the filter choice change the answer? then plot
inst = taillard["ta01"]
for label, rule in RULES.items():
    with_filters = makespan_of(inst, rule)
    no_filters = makespan_of(inst, rule, ready_operations_filter=None)
    print(f"{label}: default filters {with_filters}, no filters {no_filters}")

baselines.set_index("name")[list(RULES)].plot(kind="bar", figsize=(9, 4))
plt.plot(range(len(baselines)), baselines["ref"], "k_", markersize=20, label="optimum")
plt.ylabel("makespan")
plt.legend()
plt.tight_layout()
plt.show()